In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim


class DQNAgent:
    def __init__(
        self,
        state_size,
        action_size,
        model_class,
        buffer,
        gamma=0.99,
        lr=0.001,
        batch_size=64,
        epsilon=1.0,
        epsilon_min=0.01,
        epsilon_decay=0.995,
        tau=0.005,
    ):
        self.state_size   = state_size
        self.action_size  = action_size
        self.memory       = buffer
        self.gamma        = gamma
        self.batch_size   = batch_size
        self.epsilon      = epsilon
        self.epsilon_min  = epsilon_min
        self.epsilon_decay= epsilon_decay
        self.tau          = tau
        self.device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q_network      = model_class(state_size, action_size).to(self.device)
        self.target_network = model_class(state_size, action_size).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.criterion = nn.MSELoss()

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        self.q_network.eval()
        with torch.no_grad():
            q_values = self.q_network(state)
        self.q_network.train()
        return torch.argmax(q_values).item()

    def store(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)

    def learn(self):
        if not self.memory.is_ready():
            return None                          # ← FIXED: return None, not nothing

        states, actions, rewards, next_states, dones = self.memory.sample()

        states      = states.to(self.device).float()
        actions     = actions.unsqueeze(1).to(self.device).long()
        rewards     = rewards.unsqueeze(1).to(self.device).float()
        next_states = next_states.to(self.device).float()
        dones       = dones.unsqueeze(1).to(self.device).float()

        # Double DQN target
        with torch.no_grad():
            next_actions = torch.argmax(self.q_network(next_states), dim=1, keepdim=True)
            next_q       = self.target_network(next_states).gather(1, next_actions)
            target_q     = rewards + (1 - dones) * self.gamma * next_q

        current_q = self.q_network(states).gather(1, actions)
        loss      = self.criterion(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()

        self.soft_update()
        return loss.item()                       # ← FIXED: return scalar loss

    def soft_update(self):
        for t_param, q_param in zip(self.target_network.parameters(), self.q_network.parameters()):
            t_param.data.copy_(self.tau * q_param.data + (1 - self.tau) * t_param.data)

    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
